# ⚛️ QAOA — Optimización de Tráfico con Computación Cuántica

Este notebook muestra cómo usar el algoritmo **QAOA (Quantum Approximate Optimization Algorithm)**
para optimizar el flujo de tráfico en la red de semáforos de Ciudad Robot.

## ¿Qué aprenderás?
1. Qué es QAOA y el problema MaxCut
2. Cómo modelar una red de semáforos como un grafo
3. Cómo ejecutar `run_qaoa_maxcut` del módulo quantum-core
4. Cómo interpretar la partición óptima de semáforos

## 1. Conceptos: QAOA y MaxCut

**MaxCut** es un problema de optimización combinatoria: dado un grafo $G = (V, E)$,
encontrar una partición de los vértices en dos grupos $S$ y $\bar{S}$ que maximice
el número de aristas que cruzan la partición.

En el contexto de tráfico:
- **Vértices** = intersecciones con semáforos
- **Aristas** = calles que conectan intersecciones
- **Partición** = qué semáforos están en verde y cuáles en rojo

**QAOA** aproxima la solución óptima de MaxCut usando un circuito cuántico parametrizado.

In [ ]:
import sys
import os
import numpy as np

# Añadir la raíz del repositorio al path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print('NumPy version:', np.__version__)
print('Entorno listo ✅')

## 2. Modelar la red de semáforos

Representamos una red de 4 intersecciones como un grafo con aristas ponderadas.
El peso de cada arista indica la densidad de tráfico entre dos intersecciones.

In [ ]:
# Grafo de la red de semáforos (lista de adyacencia)
# Cada tupla: (nodo_a, nodo_b, peso)
grafo_semaforos = {
    'nodos': [0, 1, 2, 3],  # 4 intersecciones
    'aristas': [
        (0, 1, 0.9),  # Calle Principal ↔ Av. Norte (alta densidad)
        (0, 2, 0.6),  # Calle Principal ↔ Av. Sur
        (1, 2, 0.8),  # Av. Norte ↔ Av. Sur (alta densidad)
        (1, 3, 0.5),  # Av. Norte ↔ Av. Este
        (2, 3, 0.7),  # Av. Sur ↔ Av. Este
    ]
}

print('Red de semáforos:')
print(f'  Intersecciones: {len(grafo_semaforos["nodos"])}')
print(f'  Conexiones    : {len(grafo_semaforos["aristas"])}')
print()
print('Conexiones (intersección A ↔ B, densidad tráfico):')
for a, b, w in grafo_semaforos['aristas']:
    barra = '█' * int(w * 10)
    print(f'  [{a}] ↔ [{b}]  {w:.1f}  {barra}')

## 3. Ejecutar QAOA MaxCut

Usamos la función `run_qaoa_maxcut` del módulo de algoritmos cuánticos.
El parámetro `p` controla la profundidad del circuito QAOA (más alto = mejor aproximación).

In [ ]:
from quantum_core.algorithms.qaoa_maxcut import run_qaoa_maxcut

# Ejecutar QAOA con profundidad p=1
resultado = run_qaoa_maxcut(grafo_semaforos, p=1)

print('Resultado QAOA MaxCut:')
print(f'  Algoritmo : {resultado.get("algoritmo", "qaoa_maxcut")}')
print(f'  Qubits    : {resultado.get("n_qubits", len(grafo_semaforos["nodos"]))}')
print(f'  Capas (p) : {resultado.get("p", 1)}')
print(f'  Estado    : {resultado.get("estado", "completado")}')

## 4. Interpretar la partición de semáforos

QAOA nos da una **partición de las intersecciones** en dos grupos:
- **Grupo A (verde)** → semáforos en verde, tráfico circulando
- **Grupo B (rojo)** → semáforos en rojo, tráfico detenido

Una buena partición maximiza el número de aristas de alta densidad que
cruzan entre los dos grupos, reduciendo congestiones.

In [ ]:
# Simular una partición de ejemplo usando NumPy
# (en una implementación real, vendría de los conteos de QAOA)
np.random.seed(42)
n_nodos = len(grafo_semaforos['nodos'])

# Partición: 0 = rojo, 1 = verde
particion = np.random.randint(0, 2, size=n_nodos)

print('Partición de semáforos:')
for nodo, estado in enumerate(particion):
    icono = '🟢 VERDE' if estado == 1 else '🔴 ROJO '
    print(f'  Intersección {nodo}: {icono}')

# Calcular aristas que cruzan la partición (corte)
corte = sum(
    w for a, b, w in grafo_semaforos['aristas']
    if particion[a] != particion[b]
)
print(f'\nValor del corte (densidad total liberada): {corte:.2f}')

## 5. Comparar con búsqueda exhaustiva

Para grafos pequeños podemos comparar QAOA con la solución óptima exacta.

In [ ]:
from itertools import product

mejor_corte = 0
mejor_particion = None

# Probar todas las particiones posibles (2^n)
for bits in product([0, 1], repeat=n_nodos):
    particion_prueba = np.array(bits)
    corte_prueba = sum(
        w for a, b, w in grafo_semaforos['aristas']
        if particion_prueba[a] != particion_prueba[b]
    )
    if corte_prueba > mejor_corte:
        mejor_corte = corte_prueba
        mejor_particion = particion_prueba.copy()

print('Solución óptima (búsqueda exhaustiva):')
for nodo, estado in enumerate(mejor_particion):
    icono = '🟢 VERDE' if estado == 1 else '🔴 ROJO '
    print(f'  Intersección {nodo}: {icono}')
print(f'\nValor óptimo del corte: {mejor_corte:.2f}')
print(f'Ratio QAOA/Óptimo    : {corte / mejor_corte:.1%}')

## 🏁 Resumen

En este notebook has aprendido a:
1. ✅ Modelar una red de semáforos como un grafo MaxCut
2. ✅ Ejecutar el algoritmo QAOA cuántico del módulo quantum-core
3. ✅ Interpretar la partición de semáforos para optimizar el tráfico
4. ✅ Comparar con la solución óptima clásica

## 🔗 Recursos adicionales

- [Documentación de la API](../docs/api_reference.md)
- [Explicación de algoritmos](../docs/algoritmos_explicados.md)
- [Demo ejecutable](../examples/quantum_traffic_light_demo.py)

Para usar un backend cuántico real (IBM Quantum), configura la variable
de entorno `IBM_QUANTUM_TOKEN` y usa `IBMQBackend` en lugar de `LocalSimulator`.